# Precision V2 — Bigger models, longer training, ensemble of seeds
Trains the 4 precision architectures with ~50% more parameters, 1000 epochs,
context-noise augmentation, and 3 different seeds.
Result: 4 architectures x 3 seeds = 12 models. The ensemble median should
match the data nearly exactly.

In [ ]:
EPOCHS = 1000
BATCH_SIZE = 8
LR = 2e-4
N_SEEDS = 3
CTX_NOISE = 0.02

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--index-url', 'https://download.pytorch.org/whl/cu121',
                       'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
AUX_DIR = list(INPUT.rglob('combined_data.csv'))[0].parent
CODE_DIR = list(INPUT.rglob('scripts'))[0].parent
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'TaylorCouetteML'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(CODE_DIR, REPO_DIR)
os.chdir(REPO_DIR)
INPUT_CSV = REPO_DIR / 'data' / 'Input' / 'combined_data.csv'
INPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUX_DIR / 'combined_data.csv', INPUT_CSV)
print('Setup ready, REPO_DIR =', REPO_DIR)

In [ ]:
OUT_DIR = WORK / 'runs' / 'precision_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
args = [sys.executable, 'scripts/train_precision_ensemble.py',
        '--epochs', str(EPOCHS),
        '--batch', str(BATCH_SIZE),
        '--lr', str(LR),
        '--n_seeds', str(N_SEEDS),
        '--ctx_noise', str(CTX_NOISE),
        '--out_root', str(OUT_DIR)]
print('>>>', ' '.join(args))
t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO_DIR))
print(f'<<< exit={rc} elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'training failed'

In [ ]:
for root, dirs, files in os.walk(OUT_DIR):
    for f in files:
        p = Path(root) / f
        if p.suffix in ['.pt', '.pth', '.json', '.csv', '.npz']:
            print(p.relative_to(WORK), f'({p.stat().st_size/1e6:.2f} MB)')